# Strategy 14. Use automatically generated schema in naive strategy

Database schema generated by LangChain neo4j is added to the prompt. 

- 14a - "normal" schema (generated by LangChain) is used
- 14b - "enhanced" schema is used

In [1]:
import os
from dotenv import load_dotenv
from tqdm import tqdm
import json
import pandas as pd

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")


Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Wrappers for LLMs and KG

In [2]:
# Langchain wrappers for different models

import re
from langchain_openai import ChatOpenAI
from langchain_mistralai import ChatMistralAI
from langchain_anthropic import ChatAnthropic

chat_models = {}

def new_ChatModel(model):
    if re.search(r"^gpt", model):
        return ChatOpenAI(model = model)
    elif re.search(r"^o1", model):
        return ChatOpenAI(model = model, temperature = 1)
    elif re.search(r"^claude", model):
        return ChatAnthropic(model = model)
    elif re.search(r"mistral", model):
        return ChatMistralAI(model = model)
    else:
        raise ValueError(f"Unsupported model: {model}")

def ChatModel(model):
    if model in chat_models:
        return chat_models[model]
    else:
        chat_models[model] = new_ChatModel(model)
        return chat_models[model]



In [3]:
# Cypher data extraction

def extract_cypher(message):
    text = message
    pattern = r"```cypher(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)
    try:
        return [match.strip() for match in matches]
    except Exception:
        raise ValueError(f"Failed to parse: {message}")

from py2neo import Graph

graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)



In [4]:
# Some queries take a very long time to run. This will deal with timeout

import threading

class TimeoutThread(threading.Thread):
    def __init__(self, func, *args, **kwargs):
        threading.Thread.__init__(self)
        self.func = func
        self.args = args
        self.kwargs = kwargs
        self.result = None
        self.error = None

    def run(self):
        try:
            self.result = self.func(*self.args, **self.kwargs)
        except Exception as e:
            self.error = e

def run_with_timeout(func, timeout, *args, **kwargs):
    thread = TimeoutThread(func, *args, **kwargs)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        raise TimeoutError("Function execution timed out")
    elif thread.error:
        raise thread.error
    return thread.result

In [5]:
# convenience functions for data retrieval from graph

import time

def query_cypher_graph(graph, query):
    return graph.query(query)

def query_graph(llm_output):
    try:
        cypher_query = extract_cypher(llm_output)
    except Exception as e:
        return [{
            "query":None,
            "success":False,
            "exception":str(e)
        }]

    cypher_results = []
    for query in cypher_query:
        try:
            start = time.time()
            result = run_with_timeout(query_cypher_graph, 20, graph, query)
            duration = time.time() - start
            cypher_results.append({
                "query":query,
                "success":True,
                "result": list(result),
                "time": duration
                })
        except Exception as e:
            cypher_results.append({
                "query":query,
                "success":False,
                "exception":str(e)
            })
    return cypher_results


def process_results(todo, llm_answers, cypher_results):
    results = []
    for t,llm,res in zip(todo, llm_answers, cypher_results):
        out = {
            "model" : t[0],
            "question" : t[1],
            "llm_answer" : llm,
            "cypher_output": res,
            "n_cypher_queries" : len(res)
        }
        if len(res) > 0:
            out.update({
                "query": res[0].get('query',''),
                "success": res[0]['success']
            })
            if res[0]['success']:
                out.update({
                    "results": res[0]['result'],
                    "time": res[0]['time'],
                    "count" : len(res[0]['result'])
                })
            else:
                out.update({
                    "error": res[0]['exception']
                })
        results.append(out)
    return results



## Questions

In [6]:
questions = [
    "What (or how strong, or is there any) is the evidence between TDP-43 and amyotrophic lateral sclerosis (ALS)",
    "What is the evidence linking TDP-43 to cancer in animal models?",
    "What (or is there) is the clinical evidence linking BRAF to Melanoma?"
]

## Running template-based query

- 14a - normal schema
- 14b - enchanced schema


In [7]:
# Graph schema

from langchain_community.graphs import Neo4jGraph

graph = Neo4jGraph(url=neo4j_uri, username=neo4j_username, password=neo4j_password)
graph.refresh_schema()

normal_schema = graph.schema

enhanced_graph = Neo4jGraph(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_password,
    enhanced_schema=True,
)

enhanced_schema = enhanced_graph.schema

print(enhanced_schema)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The procedure has a deprecated field. ('config' used by 'apoc.meta.graphSample' is deprecated.)} {position: line: 1, column: 1, offset: 0} for query: "CALL apoc.meta.graphSample() YIELD nodes, relationships RETURN nodes, [rel in relationships | {name:apoc.any.property(rel, 'type'), count: apoc.any.property(rel, 'count')}] AS relationships"


Node properties:
- **Entity**
  - `literature`: LIST Min Size: 1, Max Size: 1
  - `score`: FLOAT Example: "0.03"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Example: "obi"
  - `approvedSymbol`: STRING 
  - `biotype`: STRING 
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
- **ThingWithTaxon**
  - `code`: STRING Example: "http://purl.obolibrary.org/obo/OBI_1110122"
  - `name`: STRING Example: "pathological process"
  - `description`: STRING Example: "Abnormal, harmful processes caused by or associate"
  - `source`: STRING Example: "Open Targets"
  - `licence`: STRING Example: "https://platform-docs.opentargets.org/licence"
  - `version`: STRING Example: "22.11"
  - `id`: STRING Example: "obi:1110122"
  - `preferred_id`: STRING Ex

In [8]:
open("normal_schema.txt", "w").write(normal_schema)
open("enhanced_schema.txt", "w").write(enhanced_schema)

25642

In [10]:
from langchain_core.prompts import PromptTemplate

system_prompt_generic = """
You are a biological data scientist with vast experience in building and extracting information from knowledge graph. 

Your current project is to support scientists who want to answer scientific questions about genes, diseases and drugs. You have a Neo4j database with biological data that accepts queries cypher queries. Scientists will provide you questions, and you should output correct cypher statements that will generate results. 

{schema}

"""

normal_schema_description = f"""\
This is graph schema:
--------------------------------------------
{normal_schema}
--------------------------------------------    
"""

enhanced_schema_description = f"""\
This is graph schema:
--------------------------------------------
{enhanced_schema}
--------------------------------------------    
"""

system_prompt_normal_schema = system_prompt_generic.format(schema = normal_schema_description)
system_prompt_enhanced_schema = system_prompt_generic.format(schema = enhanced_schema_description)

template_query = """
{question}
"""

user_template = PromptTemplate.from_template(template_query)


# Option 14a - normal schema



In [11]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]

def run_llm_14a(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_normal_schema}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_normal_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None

llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_14a(llm_model, question))

Prompting LLM:  54%|█████▍    | 65/120 [14:19<02:33,  2.79s/it]  

Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"message":"Requests rate limit exceeded"}


Prompting LLM: 100%|██████████| 120/120 [30:53<00:00, 15.45s/it]


In [12]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_14a(llm_model, question)
        time.sleep(2)

In [13]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:   0%|          | 0/120 [00:00<?, ?it/s]Failed to read from defunct connection IPv4Address(('pistoia.neo4j.rbsapp.net', 7687)) (ResolvedIPv4Address(('148.113.161.114', 7687)))
Transaction failed and will be retried in 1.1611445240057536s (Failed to read from defunct connection IPv4Address(('pistoia.neo4j.rbsapp.net', 7687)) (ResolvedIPv4Address(('148.113.161.114', 7687))))
Querying graph:  26%|██▌       | 31/120 [00:11<00:29,  2.98it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: TARGET)} {position: line: 1, column: 46, offset: 45} for query: "MATCH (g:Gene {appro

In [14]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("14a-evaluations.xlsx", index=False)
with open("14a-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To find the evidence between TDP-43 and amyotr...,[{'query': 'MATCH (gene:Gene {approvedSymbol: ...,1,"MATCH (gene:Gene {approvedSymbol: ""TDP-43""})-[...",True,[],2.163937,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...","To answer this question, we need to look at th...",[{'query': 'MATCH (g:Gene {approvedSymbol: 'TD...,1,MATCH (g:Gene {approvedSymbol: 'TDP-43'})-[:IS...,True,[],0.260530,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the evidence or strength of assoc...,"[{'query': 'MATCH (g:Gene {approvedSymbol: ""TD...",1,"MATCH (g:Gene {approvedSymbol: ""TDP-43""})-[:IS...",True,[],0.263926,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To find the evidence between TDP-43 and amyotr...,[{'query': 'MATCH (g:Gene)-[r:IS_PART_OF]->(a:...,1,MATCH (g:Gene)-[r:IS_PART_OF]->(a:GeneToDiseas...,True,[],0.280991,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the evidence between TDP-43 and a...,[{'query': 'MATCH (gene:Gene {approvedSymbol: ...,1,"MATCH (gene:Gene {approvedSymbol: ""TARDBP""})-[...",True,[],0.261642,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': 'MATCH (gene:Gene {approvedSymbol: ...,2,MATCH (gene:Gene {approvedSymbol: 'BRAF'})\n--...,True,[],0.268877,0.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,"To answer the question ""What (or is there) is ...",[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,2,MATCH (g:Gene {approvedSymbol: 'BRAF'})--(a:Ge...,True,"[{'Literature': None, 'Score': 0.2, 'Source': ...",1.504773,5847.0,NaN
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking the gene...,[{'query': 'MATCH (g:Gene {approvedSymbol: 'BR...,1,MATCH (g:Gene {approvedSymbol: 'BRAF'})\n-[:IS...,True,[],0.334867,0.0,NaN
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking the gene...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(a:G...,1,MATCH (g:Gene)-[:IS_PART_OF]->(a:GeneToDisease...,True,[],0.340567,0.0,NaN


# Option 14b - with enhanced graph schema


In [15]:
from langchain.schema import AIMessage, HumanMessage, SystemMessage

models = ["gpt-4o", "claude-3-5-sonnet-20240620", "open-mistral-7b", "o1-preview-2024-09-12"]
niter = 10

todo = [(m, q) for q in questions for m in models for _ in range(niter)]


def run_llm_14b(llm_model, question):
    try:
        llm = ChatModel(model = llm_model)
        user_prompt = user_template.invoke({"question": question})

        # o1-preview does not support system messages, so joining them together:
        if llm_model == 'o1-preview-2024-09-12':
            messages = f"{system_prompt_enhanced_schema}\n------------------------------------------\nUser question:\n{user_prompt.text}\n"
            result = llm.invoke(messages)
        else:
            messages = [
                SystemMessage(content=system_prompt_enhanced_schema),
                HumanMessage(content=user_prompt.text)
            ]
            result = llm.invoke(messages)
        return result.content
    except Exception as e:
        print(e)
        return None


llm_answers = []
for llm_model, question in tqdm(todo, desc="Prompting LLM"):
    llm_answers.append(run_llm_14b(llm_model, question))

Prompting LLM: 100%|██████████| 120/120 [28:19<00:00, 14.16s/it] 


In [17]:
import time
for i, (llm_model, question) in enumerate(todo):
    if not llm_answers[i]:
        llm_answers[i] = run_llm_14b(llm_model, question)
        time.sleep(2)

In [18]:
cypher_results = []
for answer in tqdm(llm_answers, desc="Querying graph"):
    cypher_results.append(query_graph(answer))

Querying graph:  29%|██▉       | 35/120 [00:19<01:06,  1.27it/s]Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownRelationshipTypeWarning} {category: UNRECOGNIZED} {title: The provided relationship type is not in the database.} {description: One of the relationship types in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing relationship type is: ASSOCIATED_WITH)} {position: line: 7, column: 9, offset: 128} for query: "MATCH \n  (g:Gene { approvedSymbol: 'TARDBP' }),\n  (d)\nWHERE \n  d.name =~ '(?i).*amyotrophic lateral sclerosis.*'\nMATCH \n  (g)-[:ASSOCIATED_WITH]->(a:GeneToDiseaseAssociation)-[:ASSOCIATED_WITH]->(d)\nRETURN \n  a.score AS Score,\n  a.source AS Source,\n  a.literature AS Literature,\n  a.id AS AssociationID\nORDER BY \n  Score DESC"
Received notification from DBMS server: {severity: WARNING} {c

In [19]:
results = process_results(todo, llm_answers, cypher_results)

results_df = pd.DataFrame(results)
results_df.to_excel("14b-evaluations.xlsx", index=False)
with open("14b-evaluations.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4)
results_df

,model,question,llm_answer,cypher_output,n_cypher_queries,query,success,results,time,count,error
0,gpt-4o,"What (or how strong, or is there any) is the e...",To find the evidence between TDP-43 and amyotr...,[{'query': 'MATCH (gene:Gene {approvedSymbol: ...,1,"MATCH (gene:Gene {approvedSymbol: ""TDP-43""})-[...",True,[],0.260419,0.0,NaN
1,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the strength of evidence between ...,"[{'query': 'MATCH (g:Gene {approvedSymbol: ""TD...",1,"MATCH (g:Gene {approvedSymbol: ""TDP-43""})-[:IS...",True,[],0.273702,0.0,NaN
2,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the strength of evidence between ...,[{'query': 'MATCH (g:Gene)-[:IS_PART_OF]->(ass...,1,MATCH (g:Gene)-[:IS_PART_OF]->(assoc:GeneToDis...,True,[],0.263012,0.0,NaN
3,gpt-4o,"What (or how strong, or is there any) is the e...",To retrieve the evidence between TDP-43 and am...,[{'query': 'MATCH (gene:Gene)-[assoc:GeneToDis...,1,MATCH (gene:Gene)-[assoc:GeneToDiseaseAssociat...,True,[],0.266622,0.0,NaN
4,gpt-4o,"What (or how strong, or is there any) is the e...",To determine the evidence between TDP-43 and a...,[{'query': 'MATCH (g:Gene)-[r:IS_PART_OF]->(a:...,1,MATCH (g:Gene)-[r:IS_PART_OF]->(a:GeneToDiseas...,True,[],0.267968,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...
115,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': 'MATCH (gene:HumanGene {approvedSym...,7,MATCH (gene:HumanGene {approvedSymbol: 'BRAF'}...,True,[],0.273524,0.0,NaN
116,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking the gene...,[{'query': 'MATCH (gene:Gene {approvedSymbol...,1,"MATCH\n (gene:Gene {approvedSymbol: ""BRAF""})-...",False,NaN,NaN,NaN,{code: Neo.ClientError.Statement.SyntaxError} ...
117,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,```cypher\nMATCH \n (gene:HumanGene {approved...,[{'query': 'MATCH (gene:HumanGene {approved...,1,MATCH \n (gene:HumanGene {approvedSymbol: 'BR...,False,NaN,NaN,NaN,{code: Neo.ClientError.Statement.SyntaxError} ...
118,o1-preview-2024-09-12,What (or is there) is the clinical evidence li...,To find the clinical evidence linking **BRAF**...,[{'query': 'MATCH (gene:HumanGene {approvedSym...,1,MATCH (gene:HumanGene {approvedSymbol: 'BRAF'}...,True,[],0.272721,0.0,NaN
